# Phase 4 & 5 : Modélisation Classique et Machine Learning (ML)

Ce notebook présente la phase de modélisation classique par apprentissage automatique pour prédire les arrivées touristiques au Maroc.

## Modèles entraînés et évalués :
1. **Régression Ridge** : Linéaire régularisée L2.
2. **SVR (Support Vector Regression)** : Modèle non linéaire robuste avec noyau RBF.
3. **XGBoost (eXtreme Gradient Boosting)** : Algorithme d'arbres décisionnels boostés.
4. **LightGBM (Light Gradient Boosting Machine)** : Modèle d'arbres ultra-rapide.
5. **CatBoost (Categorical Boosting)** : Stable et performant.
6. **Modèle Hybride (Forecast Hybride)** : Combinaison pondérée de prédictions (ex: Ridge + XGBoost + CatBoost) pour maximiser la robustesse.

## Explicabilité (Explainable AI - XAI) :
- Analyse de l'importance globale des caractéristiques (*Feature Importance*).
- Calcul et affichage des **valeurs SHAP (SHAP values)** pour comprendre la contribution exacte des variables clés (REER, Prix du Pétrole, Ramadan, haute saison).

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ajout du dossier parent au chemin système pour importer src
sys.path.append(os.path.abspath('..'))

from src.config import DATA_DIR, TARGET_COL, TEST_START_DATE
import src.models_ml as ml
import src.evaluation as eval_mod
import src.visualization as viz

## 1. Chargement des Données Préparées (Splits Train/Test)

In [ ]:
separated_dir = os.path.join(DATA_DIR, 'separted')

X_train = pd.read_csv(os.path.join(separated_dir, 'X_train.csv'))
y_train = pd.read_csv(os.path.join(separated_dir, 'y_train.csv')).squeeze()
X_test = pd.read_csv(os.path.join(separated_dir, 'X_test.csv'))
y_test = pd.read_csv(os.path.join(separated_dir, 'y_test.csv')).squeeze()

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape} | y_test shape: {y_test.shape}")

## 2. Entraînement et Recherche d'Hyperparamètres (GridSearchCV)

Chaque modèle est encapsulé dans une `Pipeline` scikit-learn avec imputation des valeurs manquantes (`SimpleImputer`) et mise à l'échelle (`StandardScaler`) pour éviter le *data leakage*.

In [ ]:
print("--- 1. Entraînement de la Régression Ridge ---")
ridge_grid = ml.train_ridge(X_train, y_train, cv=3)

print("--- 2. Entraînement du SVR ---")
svr_grid = ml.train_svr(X_train, y_train, cv=3)

print("--- 3. Entraînement d'XGBoost ---")
xgb_grid = ml.train_xgboost(X_train, y_train, cv=3)

print("--- 4. Entraînement de LightGBM ---")
lgb_grid = ml.train_lightgbm(X_train, y_train, cv=3)

print("--- 5. Entraînement de CatBoost ---")
cat_grid = ml.train_catboost(X_train, y_train, cv=3)

## 3. Forecast Hybride (Modèle d'Ensemble Pondéré)

Nous créons un modèle de prévision hybride en combinant de manière pondérée les forces des différents algorithmes (par exemple : 50% XGBoost, 30% CatBoost, et 20% Ridge).

In [ ]:
predictions = {}

# Récupération des prédictions individuelles
predictions['Ridge'] = ridge_grid.predict(X_test)
predictions['SVR'] = svr_grid.predict(X_test)

if xgb_grid:
    predictions['XGBoost'] = xgb_grid.predict(X_test)
if lgb_grid:
    predictions['LightGBM'] = lgb_grid.predict(X_test)
if cat_grid:
    predictions['CatBoost'] = cat_grid.predict(X_test)

# Construction du modèle Hybride
weights = {'XGBoost': 0.5, 'CatBoost': 0.3, 'Ridge': 0.2}
hybrid_forecaster = ml.HybridForecaster(weights=weights)
predictions['Hybrid Model'] = hybrid_forecaster.predict(predictions)

# Comparaison des modèles
results_df = eval_mod.compare_models(predictions, y_test)
display(results_df)

## 4. Visualisation des Résultats

Comparaison graphique des prédictions des modèles sur l'ensemble de test.

In [ ]:
# Récupération des dates de test
df_full = pd.read_csv(os.path.join(DATA_DIR, 'merged_tourism_data_final.csv'))
df_full['Date'] = pd.to_datetime(df_full['Date'])
test_dates = df_full[df_full['Date'] >= TEST_START_DATE]['Date'].values[:len(y_test)]

viz.plot_predictions_comparison(y_test, predictions, test_dates, title="Comparaison des Modèles ML & Hybride")

## 5. Explicabilité de l'IA (Explainable AI - XAI)

Pour valider scientifiquement nos prédictions devant un jury, nous analysons :
1. L'importance globale des caractéristiques (Feature Importance).
2. L'impact détaillé et individuel de chaque variable via les valeurs SHAP.

In [ ]:
# A. Importance des caractéristiques (ex. XGBoost)
if xgb_grid:
    importance = ml.get_feature_importance(xgb_grid.best_estimator_, X_train.columns)
    viz.plot_feature_importance(importance, title="Importance des Caractéristiques (XGBoost)")

In [ ]:
# B. Valeurs SHAP sur XGBoost (Interprétation globale et locale)
if xgb_grid:
    shap_values, X_test_trans = ml.calculate_shap_values(xgb_grid.best_estimator_, X_train, X_test)
    if shap_values is not None:
        viz.plot_shap_summary(shap_values, X_test_trans, X_train.columns)

## 6. Export des Métriques de Performance

In [ ]:
metrics_output_path = os.path.join(DATA_DIR, 'model_performance_metrics_ML.csv')
results_df.to_csv(metrics_output_path, index=False)
print(f"Métriques de performance ML sauvegardées dans : {metrics_output_path}")